# Hash Transcript to a Quotient Shift in E8^k / 2E8^k

This notebook implements the first stage of the signing flow: hash `(message, public key, salt)` to basis coordinates, map those coordinates into a direct sum lattice `E8^k`, and reduce modulo `2E8^k` to obtain a canonical quotient representative `t`.

In [1]:
from sage.all import vector, matrix, ZZ, codes
import itertools
import hashlib
import os

In [2]:
C = codes.BinaryReedMullerCode(1, 3)
CODEWORD_TUPLES = [tuple(int(x) for x in cw) for cw in C]

# Construction A setup for E8 = RM(1,3) + 2 Z^8
I2 = [vector(ZZ, [2 if i == j else 0 for j in range(8)]) for i in range(8)]
G = C.generator_matrix()
Grows = [vector(ZZ, [int(x) for x in G.row(i)]) for i in range(G.nrows())]
E8_BASIS = matrix(ZZ, I2 + Grows).row_module().basis_matrix()


def e8_mod_2e8_representatives():
    reps = []
    for bits in itertools.product([0, 1], repeat=8):
        r = vector(ZZ, [0] * 8)
        for i, b in enumerate(bits):
            if b:
                r += E8_BASIS.row(i)
        reps.append(r)
    return reps


def direct_sum_e8_basis(k):
    if k <= 0:
        raise ValueError("k must be positive")
    dim = E8_BASIS.nrows()
    basis = matrix(ZZ, dim * k, dim * k)
    for block in range(k):
        offset = block * dim
        for i in range(dim):
            for j in range(dim):
                basis[offset + i, offset + j] = E8_BASIS[i, j]
    return basis


REPS = e8_mod_2e8_representatives()
DEFAULT_DIRECT_SUM_K = 1
DIRECT_SUM_BASIS = direct_sum_e8_basis(DEFAULT_DIRECT_SUM_K)

print("Number of single-block representatives:", len(REPS))
print("Default direct-sum basis shape:", DIRECT_SUM_BASIS.nrows(), "x", DIRECT_SUM_BASIS.ncols())

Number of single-block representatives: 256
Default direct-sum basis shape: 8 x 8


In [ ]:
def _normalize_binary_message(msg_bits):
    if isinstance(msg_bits, str):
        if any(ch not in "01" for ch in msg_bits):
            raise ValueError("bitstring must only contain '0' and '1'")
        return msg_bits

    try:
        bits = []
        for b in msg_bits:
            if int(b) not in (0, 1):
                raise ValueError
            bits.append(str(int(b)))
        return "".join(bits)
    except Exception as exc:
        raise ValueError("msg_bits must be a bitstring or iterable of bits") from exc


def _pack_bitstring(bitstring):
    bitlen = len(bitstring)
    if bitlen == 0:
        return 0, b""

    pad = (-bitlen) % 8
    padded = bitstring + ("0" * pad)
    value = int(padded, 2)
    return bitlen, value.to_bytes(len(padded) // 8, "big")


def _normalize_salt(salt):
    if isinstance(salt, bytes):
        return salt
    if isinstance(salt, str):
        s = salt[2:] if salt.startswith("0x") else salt
        if len(s) % 2 != 0:
            s = "0" + s
        try:
            return bytes.fromhex(s)
        except ValueError as exc:
            raise ValueError("salt hex string is invalid") from exc
    raise ValueError("salt must be bytes or hex string")


def _normalize_pubkey(pubkey_bytes):
    if pubkey_bytes is None:
        return b""
    if isinstance(pubkey_bytes, bytes):
        return pubkey_bytes
    if isinstance(pubkey_bytes, str):
        s = pubkey_bytes[2:] if pubkey_bytes.startswith("0x") else pubkey_bytes
        if len(s) % 2 != 0:
            s = "0" + s
        try:
            return bytes.fromhex(s)
        except ValueError as exc:
            raise ValueError("pubkey hex string is invalid") from exc
    raise ValueError("pubkey_bytes must be None, bytes, or hex string")


def build_hash_transcript(
    msg_bits, salt, pubkey_bytes=None, domain_sep=b"E8-quotient-shift-v1"
):
    bitstring = _normalize_binary_message(msg_bits)
    bitlen, packed = _pack_bitstring(bitstring)
    salt_b = _normalize_salt(salt)
    pub_b = _normalize_pubkey(pubkey_bytes)
    transcript = (
        domain_sep
        + len(pub_b).to_bytes(4, "big") + pub_b
        + len(salt_b).to_bytes(2, "big") + salt_b
        + bitlen.to_bytes(8, "big") + packed
    )
    return transcript, salt_b


def hash_to_basis_coefficients(
    msg_bits,
    salt,
    pubkey_bytes=None,
    basis=None,
    k=DEFAULT_DIRECT_SUM_K,
    coeff_bytes=4,
):
    if coeff_bytes <= 0:
        raise ValueError("coeff_bytes must be positive")
    if basis is None:
        basis = direct_sum_e8_basis(k)

    transcript, salt_b = build_hash_transcript(msg_bits, salt, pubkey_bytes=pubkey_bytes)
    dim = basis.nrows()
    stream = hashlib.shake_256(transcript).digest(dim * coeff_bytes)
    coeffs = []
    for i in range(dim):
        start = i * coeff_bytes
        chunk = stream[start:start + coeff_bytes]
        coeffs.append(ZZ(int.from_bytes(chunk, "big")))
    return vector(ZZ, coeffs), salt_b


def basis_coefficients_to_lattice_point(coeffs, basis=None, k=DEFAULT_DIRECT_SUM_K):
    if basis is None:
        basis = direct_sum_e8_basis(k)
    coeff_vec = vector(ZZ, coeffs)
    if len(coeff_vec) != basis.nrows():
        raise ValueError("coefficient vector length must match the number of basis rows")
    return coeff_vec * basis


def reduce_coefficients_mod_2e8(coeffs, basis=None, k=DEFAULT_DIRECT_SUM_K):
    if basis is None:
        basis = direct_sum_e8_basis(k)
    coeff_vec = vector(ZZ, coeffs)
    quotient_bits = vector(ZZ, [ZZ(c % 2) for c in coeff_vec])
    rep = quotient_bits * basis
    lift_coeffs = vector(ZZ, [(coeff_vec[i] - quotient_bits[i]) // 2 for i in range(len(coeff_vec))])
    return quotient_bits, rep, lift_coeffs


def hash_message_to_shift(
    msg_bits,
    salt=None,
    pubkey_bytes=None,
    basis=None,
    k=DEFAULT_DIRECT_SUM_K,
    coeff_bytes=4,
):
    if salt is None:
        salt_b = os.urandom(16)
    else:
        salt_b = _normalize_salt(salt)

    coeffs, salt_b = hash_to_basis_coefficients(
        msg_bits,
        salt_b,
        pubkey_bytes=pubkey_bytes,
        basis=basis,
        k=k,
        coeff_bytes=coeff_bytes,
    )
    x = basis_coefficients_to_lattice_point(coeffs, basis=basis, k=k)
    quotient_bits, t, lift_coeffs = reduce_coefficients_mod_2e8(coeffs, basis=basis, k=k)
    if basis is None:
        basis = direct_sum_e8_basis(k)
    x_minus_t = x - t
    return {
        "salt": salt_b,
        "k": k,
        "basis": basis,
        "coeffs": coeffs,
        "x": x,
        "quotient_bits": quotient_bits,
        "lift_coeffs": lift_coeffs,
        "t": t,
        "x_minus_t": x_minus_t,
        "same_coset_mod_2E8": x_minus_t == 2 * (lift_coeffs * basis),
    }


In [ ]:
# transcript -> basis coefficients -> E8^k point -> quotient representative t
message = "10100100101101100001"
fixed_salt = bytes.fromhex("00112233445566778899aabbccddeeff")
pubkey = bytes.fromhex("aabbccddeeff00112233445566778899")
k = 64

data = hash_message_to_shift(message, salt=fixed_salt, pubkey_bytes=pubkey, k=k)
print("direct-sum rank:", data["basis"].nrows())
print("ambient dimension:", data["basis"].ncols())
print("salt (hex):", data["salt"].hex())
print("hashed basis coefficients:", data["coeffs"])
print("lattice point x:", data["x"])
print("quotient bits mod 2:", data["quotient_bits"])
print("quotient representative t:", data["t"])
print("x - t:", data["x_minus_t"])
print("x and t differ by an element of 2E8^k:", data["same_coset_mod_2E8"])


# Determinism for fixed (message, pubkey, salt)
data_2 = hash_message_to_shift(message, salt=fixed_salt, pubkey_bytes=pubkey, k=k)
print("\nDeterministic with fixed transcript inputs:", data["t"] == data_2["t"] and data["x"] == data_2["x"])


# Public key changes the hashed lattice point
other_pubkey = bytes.fromhex("11111111111111111111111111111111")
data_3 = hash_message_to_shift(message, salt=fixed_salt, pubkey_bytes=other_pubkey, k=k)
print("Different public key changes x:", data["x"] != data_3["x"])


direct-sum rank: 512
ambient dimension: 512
salt (hex): 00112233445566778899aabbccddeeff
hashed basis coefficients: (4027457515, 693410371, 2904650313, 3223569684, 1474406776, 2631429212, 3060123736, 3500575412, 4163292358, 2571749372, 1373344284, 2140721189, 2312423671, 2821626774, 1074085629, 1087500400, 362826057, 2753069051, 1915886186, 3368729499, 2722297135, 4198686239, 1160211408, 2154286529, 2723628973, 2612363808, 4003942754, 994096734, 3226519190, 1131448575, 3492643143, 3604358345, 1291529817, 768922193, 2078082111, 3635838449, 2140930507, 3405881233, 1342408405, 3866272940, 611114558, 3408145525, 1328377559, 3188413441, 1505716553, 3738374326, 25133957, 4046299352, 3363284516, 3334988873, 442224653, 3666097872, 2797312965, 4011479095, 1083024872, 2328657929, 3581618116, 2265238143, 4092337950, 3670947214, 2924218780, 2604931933, 1003587959, 2501712781, 1333400029, 3892612469, 1396335342, 3525469986, 2475248522, 3908240823, 78376833, 2112796036, 1463072632, 3328350978, 20762